# 🎬 MiniMax H3 Video Generation on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/keboqi/minimax-h3/blob/main/minimax_h3_colab.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue.svg)](https://github.com/keboqi/minimax-h3)

Run MiniMax H3 on the **Colab G4 GPU** (NVIDIA RTX PRO 6000 Blackwell, 96 GB VRAM). The first launch installs the Python and ComfyUI environment. The MiniMax H3 default models download in the background while the app starts; tick more groups in the launch cell, and anything else downloads on first use.

The notebook can mount Google Drive for persistent outputs and keep model checkpoints in system RAM when `/dev/shm` has room.

Paths are relative to the directory containing this notebook. Start the kernel there, or set `NOTEBOOK_DIRECTORY` in cell 1 if its working directory differs. A local repository is reused only when its core files and all Git-tracked files are present; otherwise it is cloned into `minimax-h3/` beside the notebook. The virtual environment and optional Drive mount also live beside the notebook.

---
### Runtime
Select **G4 GPU** in Colab's runtime settings before running the first cell to use the Studio. The Workshop (media library and storyboard planning) needs no GPU, so a CPU runtime is enough for it; the hardware check then reports that only the Workshop can launch.


In [1]:
# @title 1. Verify GPU Hardware and System Resources
import shutil
import subprocess
from pathlib import Path

NOTEBOOK_DIRECTORY = ""  # @param {type:"string"}
# Capture once, before later cells change the working directory.
NOTEBOOK_DIR = Path(NOTEBOOK_DIRECTORY or globals().get("NOTEBOOK_DIR", Path.cwd())).expanduser().resolve()
if not NOTEBOOK_DIR.is_dir():
    raise FileNotFoundError(f"Notebook directory does not exist: {NOTEBOOK_DIR}")
print(f"Notebook directory: {NOTEBOOK_DIR}")

try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True,
    ).strip().splitlines()
except (OSError, subprocess.CalledProcessError):
    gpu_info = []

# The Studio needs the G4; the Workshop runs on any runtime, CPU included.
COLAB_G4 = len(gpu_info) == 1 and "RTX PRO 6000" in gpu_info[0].upper()

def meminfo_bytes(field):
    with open("/proc/meminfo", encoding="utf-8") as info:
        for line in info:
            if line.startswith(field + ":"):
                return int(line.split()[1]) * 1024
    raise RuntimeError(f"Missing {field} in /proc/meminfo")

if COLAB_G4:
    print(f"[OK] Colab G4: {gpu_info[0]}")
else:
    print(f"[i] No Colab G4 (found: {gpu_info or 'no GPU'}). Only the Workshop can launch on this runtime; "
          "select G4 GPU in Colab's runtime settings and reconnect to use the Studio.")
print(f"System RAM available: {meminfo_bytes('MemAvailable') / 2**30:.1f} GiB")
for location in (NOTEBOOK_DIR, Path("/dev/shm")):
    space = shutil.disk_usage(location)
    print(f"{location} free: {space.free / 2**30:.1f} GiB / {space.total / 2**30:.1f} GiB")


Notebook directory: /content
[OK] Colab G4: NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB, 580.82.07
System RAM available: 174.2 GiB
/content free: 235.6 GiB / 235.7 GiB
/dev/shm free: 87.0 GiB / 87.0 GiB


### 2. Clone Repository & Setup Python 3.12 Environment
MiniMax H3 requires Python 3.12, PyTorch 2.11 + CUDA 13.0, and prebuilt acceleration wheels (such as SageAttention 2.2.0 cp312).
The next cell creates a Python 3.12 environment with `uv`, or reuses the existing Python 3.12 environment on reruns.


In [2]:
# @title 2. Clone Repository & Create Python 3.12 Virtualenv
import os
import sys
import subprocess
import uuid
import shutil
from pathlib import Path

REPO_URL = "https://github.com/keboqi/minimax-h3.git"  # @param {type:"string"}
NOTEBOOK_DIR = Path(globals().get("NOTEBOOK_DIR", Path.cwd())).resolve()

def is_complete_repository(directory):
    required_files = (
        "run_h3.sh", "setup_h3.py", "gradio_app.py", "h3_models.py",
        "h3_app/__init__.py", "h3_ui/application.py",
    )
    if not (directory / ".git").exists() or not all(
        (directory / relative_path).is_file() for relative_path in required_files
    ):
        return False
    tracked = subprocess.run(
        ["git", "-C", str(directory), "ls-tree", "-r", "--name-only", "-z", "HEAD"],
        capture_output=True, text=True,
    )
    if tracked.returncode != 0:
        return False
    tracked_files = [name for name in tracked.stdout.split("\0") if name]
    return set(required_files).issubset(tracked_files) and all(
        (directory / name).is_file() for name in tracked_files
    )

# Reuse a complete checkout beside the notebook, or check its minimax-h3 child.
workspace = NOTEBOOK_DIR if is_complete_repository(NOTEBOOK_DIR) else NOTEBOOK_DIR / "minimax-h3"
WORKSPACE_DIR = str(workspace)
VENV_DIR = str(NOTEBOOK_DIR / "venv_h3")
os.environ["UV_NO_CACHE"] = "1"
os.environ["PIP_NO_CACHE_DIR"] = "1"

# 1. Clone or update repository
cloned_repository = False
if is_complete_repository(workspace):
    print(f"[OK] Complete repository present at {WORKSPACE_DIR}; syncing latest code...")
else:
    if workspace.exists():
        # Preserve incomplete checkouts while keeping the clone destination predictable.
        backup = workspace.with_name(f"minimax-h3-incomplete-{uuid.uuid4().hex[:8]}")
        if workspace.resolve().parent != NOTEBOOK_DIR or backup.resolve().parent != NOTEBOOK_DIR:
            raise RuntimeError(f"Cannot preserve incomplete checkout outside the notebook directory: {workspace}")
        workspace.rename(backup)
        print(f"[Colab] Preserved incomplete checkout at {backup}")
    print(f"Cloning MiniMax H3 repository from {REPO_URL} into {WORKSPACE_DIR}...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORKSPACE_DIR], check=True)
    cloned_repository = True

if not cloned_repository and (workspace / ".git").exists():
    def git(*args):
        return subprocess.run(["git", "-C", WORKSPACE_DIR, *args], check=True, capture_output=True, text=True)

    if subprocess.run(["git", "-C", WORKSPACE_DIR, "remote", "get-url", "origin"], capture_output=True).returncode == 0:
        git("remote", "set-url", "origin", REPO_URL)
    else:
        git("remote", "add", "origin", REPO_URL)
    if subprocess.run(["git", "-C", WORKSPACE_DIR, "rev-parse", "--verify", "HEAD"], capture_output=True).returncode == 0:
        git("fetch", "origin", "HEAD")
        if not git("branch", "--show-current").stdout.strip():
            git("switch", "-c", f"colab-sync-{uuid.uuid4().hex[:8]}")
        git("merge", "--ff-only", "FETCH_HEAD")
    else:
        git("fetch", "--depth", "1", "origin", "HEAD")
        git("switch", "-c", "main", "FETCH_HEAD")

if not is_complete_repository(workspace):
    raise RuntimeError(f"Repository checkout is incomplete: {WORKSPACE_DIR}")

os.chdir(WORKSPACE_DIR)
print(f"Workspace directory: {WORKSPACE_DIR}")

# 2. Install uv and provision Python 3.12 only when needed.
def ensure_python_environment(venv_dir):
    venv = Path(venv_dir)
    python = venv / "bin" / "python3"
    if (venv / "pyvenv.cfg").is_file() and python.is_file():
        version = subprocess.run(
            [str(python), "-c", "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')"],
            capture_output=True, text=True,
        )
        if version.returncode != 0 or version.stdout.strip() != "3.12":
            raise RuntimeError(f"Existing environment at {venv} is not a working Python 3.12 environment. Preserve or move it before creating a replacement.")
        print(f"[OK] Reusing Python 3.12 environment at {venv}")
        create_environment = False
    else:
        if venv.exists() and (not venv.is_dir() or any(venv.iterdir())):
            raise RuntimeError(f"Existing path at {venv} is not a usable virtual environment. Preserve or move it before creating a replacement.")
        create_environment = True

    uv_executable = shutil.which("uv")
    if not uv_executable:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv>=0.8"], check=True)
        uv_executable = shutil.which("uv")
    uv_command = [uv_executable] if uv_executable else [sys.executable, "-m", "uv"]
    if create_environment:
        print(f"Creating isolated Python 3.12 environment at {venv}...")
        subprocess.run(uv_command + ["venv", "--python", "3.12", str(venv)], check=True)
    return uv_command

uv_command = ensure_python_environment(VENV_DIR)
# Remove any persistent uv cache left by earlier cells or a reused runtime.
uv_cache_env = os.environ.copy()
uv_cache_env.pop("UV_NO_CACHE", None)
subprocess.run(uv_command + ["cache", "clean"], check=True, env=uv_cache_env)

# 3. Add the venv to PATH without duplicating entries on reruns.
venv_bin = str(Path(VENV_DIR) / "bin")
path_entries = [entry for entry in os.environ["PATH"].split(os.pathsep) if entry != venv_bin]
os.environ["PATH"] = os.pathsep.join([venv_bin, *path_entries])
os.environ["VIRTUAL_ENV"] = VENV_DIR

print("\nVerifying active Python version:")
subprocess.run([str(Path(VENV_DIR) / "bin" / "python3"), "--version"], check=True)
print(f"Python executable: {Path(VENV_DIR) / 'bin' / 'python3'}")
print("\n[OK] Setup complete! Run cells 3–5, then launch with the generated Colab wrapper.")


Cloning MiniMax H3 repository from https://github.com/keboqi/minimax-h3.git into /content/minimax-h3...
Workspace directory: /content/minimax-h3
Creating isolated Python 3.12 environment at /content/venv_h3...

Verifying active Python version:
Python executable: /content/venv_h3/bin/python3

[OK] Setup complete! Run cells 3–5, then launch with the generated Colab wrapper.


### 3. Configure Hugging Face, Gemini, and OpenAI Secrets
1. Open **Secrets** (key icon in Colab's left sidebar) and add the names below with your values. Enable **Notebook access** for each secret used here.
   - `HF_TOKEN`: Hugging Face Read token for gated model downloads and Hub rate limits. Create one at https://huggingface.co/settings/tokens.
   - `GEMINI_API_KEY`: Optional Gemini key for the default prompt writer.
   - `OPENAI_API_KEY`: Optional key for the OpenAI-compatible prompt writer.
   - `OPENAI_BASE_URL`: Optional OpenAI-compatible endpoint (needed for DeepSeek and GLM). Omit it to use the official OpenAI API.
2. Run the next cell to load secrets into the runtime environment. Values are never printed or stored in notebook fields. Missing or inaccessible secrets are skipped; optional writer keys can also be entered temporarily in the app.

Reuse the same secret names in other Colab notebooks; enable Notebook access for each notebook. After changing secrets, rerun this cell and restart the app if it is already running.


In [3]:
# @title 3. Load Hugging Face, Gemini, and OpenAI Secrets
import os
from google.colab import userdata

def configure_key(name):
    try:
        value = (userdata.get(name) or "").strip()
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
        # Do not print exception details or secret values into saved cell output.
        print(f"[i] {name} unavailable; add it in Colab Secrets and enable Notebook access if needed.")
        value = ""
    if value:
        os.environ[name] = value
    else:
        # Reruns must not retain a removed or inaccessible secret from this runtime.
        os.environ.pop(name, None)
    return bool(value)

hf_ready = configure_key("HF_TOKEN")
gemini_ready = configure_key("GEMINI_API_KEY")
openai_ready = configure_key("OPENAI_API_KEY")
openai_endpoint = configure_key("OPENAI_BASE_URL")

print("[OK] Hugging Face token configured." if hf_ready else "[i] Hugging Face token not configured; gated models require one.")
print("[OK] Gemini API key configured." if gemini_ready else "[i] No Gemini API key configured; enter one in the UI to use Gemini.")
print("[OK] OpenAI API key configured." if openai_ready else "[i] No OpenAI API key configured; enter one in the UI to use the OpenAI-compatible writer.")
print("[OK] OpenAI endpoint configured." if openai_endpoint else "[i] No OpenAI endpoint configured; the official OpenAI API is used.")


[i] GEMINI_API_KEY unavailable; add it in Colab Secrets and enable Notebook access if needed.
[i] OPENAI_API_KEY unavailable; add it in Colab Secrets and enable Notebook access if needed.
[i] OPENAI_BASE_URL unavailable; add it in Colab Secrets and enable Notebook access if needed.
[OK] Hugging Face token configured.
[i] No Gemini API key configured; enter one in the UI to use Gemini.
[i] No OpenAI API key configured; enter one in the UI to use the OpenAI-compatible writer.
[i] No OpenAI endpoint configured; the official OpenAI API is used.


### 4. Google Drive outputs
Drive is off by default, so outputs stay in the current runtime. Turn on **MOUNT_GOOGLE_DRIVE** to keep outputs in `MyDrive/MiniMax-H3/` across sessions; the cell then asks for Google Drive access when needed. An existing mount is reused.

**Launching the Workshop?** Turn on **MOUNT_GOOGLE_DRIVE** first. The Workshop works on the saved outputs and storyboards, so without Drive it only sees what this runtime created and its edits are lost when the runtime ends.


In [4]:
# @title 4. Google Drive outputs
# @markdown Off by default. When on, confirm Google Drive access when prompted; outputs persist in MyDrive/MiniMax-H3/.

import os
import shutil
from pathlib import Path

MOUNT_GOOGLE_DRIVE = True  # @param {type:"boolean"}
DRIVE_OUTPUT_DIR = ""
DRIVE_MOUNT_DIR = str(NOTEBOOK_DIR / "drive")
DRIVE_OUTPUT_DIR = DRIVE_OUTPUT_DIR or str(Path(DRIVE_MOUNT_DIR) / "MyDrive" / "MiniMax-H3")

def setup_google_drive_outputs(enabled: bool = False, drive_folder: str | None = None, workspace_dir: str | None = None):
    if not enabled:
        print("[i] Google Drive persistence disabled. Outputs will be saved locally in this Colab session.")
        return

    mount_dir = Path(globals().get("NOTEBOOK_DIR", Path.cwd())) / "drive"
    if " " in str(mount_dir):
        raise ValueError("Google Drive mount paths cannot contain spaces. Set NOTEBOOK_DIRECTORY to a directory without spaces and rerun setup before enabling Google Drive.")

    workspace = Path(workspace_dir or WORKSPACE_DIR).expanduser().resolve()
    gdrive_base = Path(drive_folder or mount_dir / "MyDrive" / "MiniMax-H3").expanduser().resolve()
    output_pairs = [
        (workspace / "h3" / "ComfyUI" / "output", gdrive_base / "output"),
        (workspace / "h3" / "gradio_outputs", gdrive_base / "gradio_outputs"),
    ]

    # Validate every pair before creating directories or moving any existing files.
    for target_index, (_, target) in enumerate(output_pairs):
        target = target.resolve()
        for source_index, (local_path, _) in enumerate(output_pairs):
            local = local_path.resolve()
            if source_index == target_index and local == target:
                continue
            if local == target or target.is_relative_to(local) or local.is_relative_to(target):
                raise ValueError(f"Output directories overlap: {local_path} and {target}. Choose a separate Drive output folder.")
    for local_path, _ in output_pairs:
        if local_path.exists() and not local_path.is_dir() and not local_path.is_symlink():
            raise ValueError(f"Output path is a file, not a directory: {local_path}")

    try:
        if os.path.ismount(mount_dir) and (mount_dir / "MyDrive").is_dir():
            print(f"[i] Reusing Google Drive already mounted at {mount_dir}.")
        else:
            print(f"Mounting Google Drive at {mount_dir}...")
            from google.colab import drive
            drive.mount(str(mount_dir))
    except Exception as exc:
        print(f"[!] Could not mount Google Drive: {exc}")
        raise RuntimeError("Drive setup stopped. Fix the issue above and rerun this cell, or turn off MOUNT_GOOGLE_DRIVE to use local outputs.") from None

    for local_path, gdrive_target in output_pairs:
        # Includes an ordinary directory that already is the requested destination.
        if local_path.resolve() == gdrive_target.resolve():
            continue
        gdrive_target.mkdir(parents=True, exist_ok=True)
        local_path.parent.mkdir(parents=True, exist_ok=True)
        if local_path.is_symlink():
            local_path.unlink()
        elif local_path.is_dir():
            for item in local_path.iterdir():
                target_item = gdrive_target / item.name
                counter = 1
                while target_item.exists() or target_item.is_symlink():
                    target_item = gdrive_target / f"{item.stem}_from_colab_{counter}{item.suffix}"
                    counter += 1
                shutil.move(str(item), str(target_item))
            # Refuse to remove anything if new files appeared during migration.
            local_path.rmdir()
        local_path.symlink_to(gdrive_target, target_is_directory=True)

    print("[OK] Google Drive output folders linked successfully!")
    print(f"     ComfyUI output  -> {output_pairs[0][1]}")
    print(f"     Gradio outputs  -> {output_pairs[1][1]}")
    print("     All generated assets will persist in your Google Drive across sessions.")

setup_google_drive_outputs(MOUNT_GOOGLE_DRIVE, DRIVE_OUTPUT_DIR)


Mounting Google Drive at /content/drive...
Mounted at /content/drive
[OK] Google Drive output folders linked successfully!
     ComfyUI output  -> /content/drive/MyDrive/MiniMax-H3/output
     Gradio outputs  -> /content/drive/MyDrive/MiniMax-H3/gradio_outputs
     All generated assets will persist in your Google Drive across sessions.


### 5. (Optional) Keep Model Checkpoints in System RAM

On G4, model files that download while the app starts (the defaults ticked in the launch cell) and Qwen Image 2.1 checkpoints go to `/dev/shm` when space permits. They remain visible at their normal `h3/ComfyUI/models` paths.

This keeps tens of gigabytes off the Colab disk, which writes about 340 MB/s and lets only a few gigabytes wait to be written; on disk, those downloads stall setup and every other writer. In RAM they download at network speed and load without reading the disk.

The download check keeps 32 GiB of system RAM and 8 GiB of `/dev/shm` free, then uses disk if a file does not fit. RAM files disappear when the Colab runtime ends, and the RAM they hold is not available to other programs while the app runs.

Use the generated `run_h3_colab.sh` for both notebook and Colab Terminal launches; plain `run_h3.sh` keeps every model on disk.

In [5]:
# @title 5. (Optional) Keep Model Checkpoints in System RAM
USE_RAM_FOR_MODELS = True  # @param {type:"boolean"}

import os
import shlex
import shutil
from pathlib import Path

workspace = Path(WORKSPACE_DIR)
launcher = workspace / "run_h3_colab.sh"
# Earlier notebooks routed Qwen Image 2.1 through a generated sitecustomize; the app now does it.
shutil.rmtree(workspace / ".colab_ram_router", ignore_errors=True)

is_tmpfs = any(
    fields[1] == "/dev/shm" and fields[2] == "tmpfs"
    for line in Path("/proc/mounts").read_text().splitlines()
    if len(fields := line.split()) >= 3
)
if USE_RAM_FOR_MODELS and not is_tmpfs:
    print("[colab-ram] /dev/shm is not a tmpfs mount; using disk for models.")
    USE_RAM_FOR_MODELS = False

ram_env = ""
if USE_RAM_FOR_MODELS:
    ram_root = Path("/dev/shm/minimax-h3-models")
    ram_root.mkdir(parents=True, exist_ok=True)
    free_gib = shutil.disk_usage(ram_root).free / 1024**3
    print(f"[colab-ram] Preloaded and Qwen Image 2.1 model files will use /dev/shm when space permits "
          f"({free_gib:.1f} GiB free).")
    ram_env = f"H3_RAM_MODELS_DIR={shlex.quote(str(ram_root))} "
command = (
    "exec env -u PYTHONPATH -u MPLBACKEND UV_NO_CACHE=1 PIP_NO_CACHE_DIR=1 "
    f"{ram_env}bash {shlex.quote(str(workspace / 'run_h3.sh'))}"
)

# A separate Colab Terminal does not inherit the notebook kernel's environment.
launcher_source = (
    "#!/usr/bin/env bash\nset -euo pipefail\n"
    f"export VIRTUAL_ENV={shlex.quote(VENV_DIR)}\n"
    f"export PATH={shlex.quote(str(Path(VENV_DIR) / 'bin'))}:\"$PATH\"\n"
    '[[ -x "$VIRTUAL_ENV/bin/python3" ]] || { echo "Python environment missing; rerun notebook cell 2." >&2; exit 1; }\n'
    'app="${H3_APP:-studio}"\n'
    'if [[ "${app,,}" == studio ]]; then\n'
    '  gpu=$(nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null) || gpu=""\n'
    '  if [[ "$gpu" == *$\'\\n\'* || "${gpu^^}" != *"RTX PRO 6000"* ]]; then\n'
    '    echo "The Studio needs the Colab G4 GPU. Select G4 or set H3_APP=workshop." >&2; exit 1\n'
    '  fi\nfi\n'
    # Terminal launches serve the fp8 Drama 3 checkpoint unless told otherwise.
    'export H3_DRAMA3_SGLANG_CHECKPOINT="${H3_DRAMA3_SGLANG_CHECKPOINT:-fp8}"\n'
    + command + "\n"
)
launcher.write_text(launcher_source, encoding="utf-8")
launcher.chmod(0o755)
COLAB_LAUNCHER = str(launcher)
print(f"[OK] Colab launcher ready: bash {COLAB_LAUNCHER}")

[colab-ram] Preloaded and Qwen Image 2.1 model files will use /dev/shm when space permits (87.0 GiB free).
[OK] Colab launcher ready: bash /content/minimax-h3/run_h3_colab.sh


### 6. Launch MiniMax H3 Studio or Workshop (`bash run_h3_colab.sh`)
You can launch the app directly using the cell below, or by running `bash run_h3_colab.sh` in the Colab Terminal. Use the workspace path printed by cell 2. If an existing `minimax-h3/` checkout is incomplete, cell 2 preserves it under a backup name and clones a fresh repository into `minimax-h3/`.

```bash
cd "<workspace path printed by cell 2>"
GRADIO_SHARE=true ENABLE_CLOUDFLARE_TUNNEL=false bash run_h3_colab.sh
```

- **Studio or Workshop**: `LAUNCH_APP` defaults to `studio`, the full app with ComfyUI and models. Pick `workshop` to start the Prism Workshop instead: the media library and storyboard planning on the same folders, without ComfyUI, model downloads or GPU use. Storyboards prepared there open in the Studio, which generates their scenes. **Start the Workshop with Google Drive mounted** (cell 4) so it works on your saved outputs and keeps its edits. In the Terminal, run `H3_APP=workshop bash run_h3_colab.sh`.
- **First launch**: Installs the environment and starts the servers.
- **Default models**: While the servers start, the launcher downloads the defaults of the groups ticked in the launch cell: MiniMax H3 by default, with Drama 3 and Qwen Image 2.1 optional. Startup does not wait for them, and anything not ticked downloads on first use. In the Terminal, pass a list such as `H3_PREPARE_MODELS=h3,drama3,qwen_image bash run_h3_colab.sh`, or `none` to skip.
- **Test updates**: Stop the launch cell, enable `PULL_LATEST_CODE`, and rerun just this cell to pull the latest code from the remote before starting. The option defaults to off. Your existing environment and downloaded models are reused; no Colab runtime restart is needed. If the pull fails, startup stops so you can resolve the Git error.
- **Drama 3**: Uses the fp8 checkpoint by default. Pick `int4` in the launch cell to leave more GPU memory for other engines, or run `H3_DRAMA3_SGLANG_CHECKPOINT=int4 bash run_h3_colab.sh` in the Terminal.
- **Public URLs**: Gradio Share is enabled by default. If its relay refuses three attempts, the launcher prints a Cloudflare URL instead. Enable `ENABLE_CLOUDFLARE_TUNNEL` in the launch cell to always open one.

The launch cell streams setup and server logs, including public URLs. It stays running while the app is serving; use the cell stop button to shut it down.

In [ ]:
# @title 6. Launch MiniMax H3 Studio or Workshop
import os
import signal
import subprocess

os.chdir(WORKSPACE_DIR)

# @markdown **App:** `studio` runs generation (Gradio & ComfyUI). `workshop` manages media and plans storyboards without ComfyUI or models; start it with Google Drive mounted (cell 4).
LAUNCH_APP = "studio"  # @param ["studio", "workshop"]

# @markdown **Update before starting:** Pull the latest remote code using a fast-forward-only merge. Stop this cell before rerunning; the existing environment and model files are reused.
PULL_LATEST_CODE = False  # @param {type:"boolean"}

# @markdown **Public links:** Gradio Share is on by default. Cloudflare is off.
ENABLE_CLOUDFLARE_TUNNEL = False  # @param {type:"boolean"}
ENABLE_GRADIO_SHARE = True  # @param {type:"boolean"}
# @markdown **Studio only:** the Drama 3 checkpoint and default model downloads below.
# @markdown **Drama 3 checkpoint:** fp8 is the higher-precision release; int4 (24 GB download) leaves more GPU memory for other engines.
DRAMA3_CHECKPOINT = "fp8"  # @param ["fp8", "int4"]
# @markdown **Download default models while starting:** the app does not wait for them; anything not ticked downloads on first use.
PREPARE_MINIMAX_H3 = True  # @param {type:"boolean"}
PREPARE_DRAMA3 = False  # @param {type:"boolean"}
PREPARE_QWEN_IMAGE = False  # @param {type:"boolean"}

if LAUNCH_APP == "studio" and not globals().get("COLAB_G4", True):
    raise RuntimeError("The Studio needs the Colab G4 GPU. Select G4 GPU in Colab's runtime settings and reconnect, "
                       "or set LAUNCH_APP to workshop.")

# Re-verify Google Drive output links if enabled
if globals().get("MOUNT_GOOGLE_DRIVE", False) and "setup_google_drive_outputs" in globals():
    setup_google_drive_outputs(MOUNT_GOOGLE_DRIVE, globals().get("DRIVE_OUTPUT_DIR"))
elif LAUNCH_APP == "workshop":
    print("[!] Google Drive is not mounted. The Workshop will only see this runtime's outputs, and its edits "
          "are lost when the runtime ends. Turn on MOUNT_GOOGLE_DRIVE in cell 4 and rerun it first.", flush=True)

def run_colab_launcher(launcher):
    """Relay child output through IPython so Colab displays logs immediately."""
    command = ["bash", str(launcher)]
    print(f"Starting MiniMax H3 {LAUNCH_APP} with {launcher}...", flush=True)
    print("This cell stays running while the app is active. Stop the cell to shut it down.", flush=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["H3_APP"] = LAUNCH_APP
    env["ENABLE_CLOUDFLARE_TUNNEL"] = str(ENABLE_CLOUDFLARE_TUNNEL).lower()
    env["GRADIO_SHARE"] = str(ENABLE_GRADIO_SHARE).lower()
    env["H3_DRAMA3_SGLANG_CHECKPOINT"] = DRAMA3_CHECKPOINT
    groups = [name for name, enabled in (
        ("h3", PREPARE_MINIMAX_H3), ("drama3", PREPARE_DRAMA3), ("qwen_image", PREPARE_QWEN_IMAGE),
    ) if enabled]
    env["H3_PREPARE_MODELS"] = ",".join(groups) or "none"
    proc = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
        env=env, start_new_session=(os.name == "posix"),
    )
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        return_code = proc.wait()
        if return_code:
            raise subprocess.CalledProcessError(return_code, command)
    finally:
        try:
            if proc.poll() is None:
                # Signal the whole Linux process group so setup and servers stop too.
                try:
                    if os.name == "posix":
                        os.killpg(proc.pid, signal.SIGTERM)
                    else:
                        proc.terminate()
                except ProcessLookupError:
                    pass
                try:
                    proc.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    try:
                        if os.name == "posix":
                            os.killpg(proc.pid, signal.SIGKILL)
                        else:
                            proc.kill()
                    except ProcessLookupError:
                        pass
                    proc.wait(timeout=5)
        finally:
            proc.stdout.close()


# The app manages its public tunnel and stops it on shutdown.

if PULL_LATEST_CODE:
    print("[Colab] Pulling latest code from origin before starting...", flush=True)
    subprocess.run(["git", "-C", WORKSPACE_DIR, "pull", "--ff-only", "origin", "HEAD"], check=True)

# Ensure core repository modules are intact
if not (Path(WORKSPACE_DIR) / "h3_ui/application.py").is_file() or not (Path(WORKSPACE_DIR) / "h3_models.py").is_file():
    print("[Colab] Restoring missing repository modules from GitHub...")
    subprocess.run(["git", "-C", WORKSPACE_DIR, "fetch", "--depth", "1", "https://github.com/keboqi/minimax-h3.git", "HEAD"], check=True)
    subprocess.run(["git", "-C", WORKSPACE_DIR, "checkout", "FETCH_HEAD", "--", "h3_ui/application.py", "h3_models.py"], check=True)

run_colab_launcher(COLAB_LAUNCHER)


[i] Reusing Google Drive already mounted at /content/drive.
[OK] Google Drive output folders linked successfully!
     ComfyUI output  -> /content/drive/MyDrive/MiniMax-H3/output
     Gradio outputs  -> /content/drive/MyDrive/MiniMax-H3/gradio_outputs
     All generated assets will persist in your Google Drive across sessions.
Starting MiniMax H3 studio with /content/minimax-h3/run_h3_colab.sh...
This cell stays running while the app is active. Stop the cell to shut it down.
[h3-run] Runtime Python: /content/venv_h3/bin/python3
[h3-run] SGLang sleep: available (NVIDIA driver 580.82.07; Omni stages use host SHM)
[h3-run] Background model preparation: h3 model files, alongside setup
[h3-stats] RAM 177 GiB; dirty_ratio 2% background 1%; CPUs 48; disks nvme0n1, nvme0n2
[h3-run] ComfyUI environment is current; checking Hugging Face model versions
[h3-prepare] Preparing h3 in the background: MiniMax H3 model files (12)
[h3-prepare] up to date Minimax-h3_Singularity_ref2va_Pruned_v1.3_int8.sa

In [9]:

from pathlib import Path
import shutil

file = Path("/content/minimax-h3/h3_app/workflows/h3.py")

old = """    fused_modulation = graph.add(
        FUSED_MODULATION_NODE,
        model=Graph.out(turbo),
        enabled=True,
    )"""

new = """    fused_modulation = graph.add(
        FUSED_MODULATION_NODE,
        model=Graph.out(turbo),
        enabled=False,
    )"""

if not file.exists():
    print("❌ 未找到文件:", file)
else:
    source = file.read_text(encoding="utf-8")

    if source.count(old) == 1:
        patched = source.replace(old, new, 1)
        compile(patched, str(file), "exec")

        backup = Path(str(file) + ".bak")
        shutil.copy2(file, backup)
        file.write_text(patched, encoding="utf-8")

        print("✅ 修复成功")
        print("已关闭不兼容的 Fused Modulation")
        print("备份文件:", backup)

    elif new in source:
        print("✅ 已经关闭 Fused Modulation")
    else:
        print("⚠️ 当前源码结构与预期不同")
        print("未修改文件，请保留原始代码")


✅ 修复成功
已关闭不兼容的 Fused Modulation
备份文件: /content/minimax-h3/h3_app/workflows/h3.py.bak


In [7]:

from pathlib import Path
import shutil

file = Path("/content/minimax-h3/h3_ui/job_bindings.py")

old = "for index in (0, 1, 2, 3, 7, 9):"
new = "for index in (0, 1, 2, 3, 5, 6):"

if not file.exists():
    print("错误：找不到项目文件，检查路径")
else:
    source = file.read_text(encoding="utf-8")

    if old in source:
        # 备份原文件
        backup = file.with_suffix(".py.bak")
        shutil.copy2(file, backup)

        # 修正输出索引
        patched = source.replace(old, new, 1)

        # 检查 Python 语法
        compile(patched, str(file), "exec")

        # 写入修复
        file.write_text(patched, encoding="utf-8")

        print("✅ 修复成功")
        print("原始文件备份：", backup)
        print("生成输出索引已经修正")

    elif new in source:
        print("✅ 此处已经修复，无需重复修改")
    else:
        print("⚠️ 代码版本不同，没有修改任何内容")
        print("请检查 metadata_output 附近的源码：")
        pos = source.find("if metadata_output:")
        print(source[pos:pos+900] if pos >= 0 else "未找到对应代码")


✅ 修复成功
原始文件备份： /content/minimax-h3/h3_ui/job_bindings.py.bak
生成输出索引已经修正


## 💡 Tips & Operation Guide

### 🎛️ User Interface & ComfyUI Access
- **Gradio Interface**: Generates video, image frames, and audio soundtracks with real-time preview and multi-stage progress reporting.
- **ComfyUI Editor**: Open `<public-url>/comfyui` in your browser (or use the ComfyUI tab in the interface) to view and edit the active node graphs live.

### 💾 Persistent Outputs on Google Drive
- When **Mount Google Drive** is enabled, all assets are stored under `MyDrive/MiniMax-H3/`.
- **ComfyUI Outputs**: Direct node saves (images, audio, videos) go to `MyDrive/MiniMax-H3/output`.
- **Gradio Outputs**: Upscaled videos, generated videos, audio tracks, and gallery items go to `MyDrive/MiniMax-H3/gradio_outputs`.
- When reconnecting in future sessions, all previous files are immediately accessible in your Google Drive and appear in the UI gallery.

### G4 model settings
- **Stage offload**: Leave the default off for compact models. Selecting the BF16 text encoder enables stage offload automatically.
- **Quality**: Singularity with Turbo is the default quick-start profile. Try Quality or Original for final outputs and compare results on your own prompts.
